# Implementing ResNet18 based - Mini ResNet

This notebook will be building a ResNet model which is derived from ResNet18 model architecture. <br> This model is called the `Mini-ResNet`.

In [2]:
import tensorflow as tf
import numpy as np
import tensorflow_datasets as tfds
from tensorflow.keras.layers import Layer

### Implement the identity block

In [4]:
from tensorflow.keras.layers import Conv2D, BatchNormalization, GlobalAvgPool2D, MaxPooling2D
from tensorflow.keras.layers import Activation, Add

# Arguments:
    # filters - Filters for the convolution layer
    # kernel size - Kernel size for the convolution layer
class IdentityBlock(tf.keras.Model):
    def __init__(self, filters, kernel_size):
        # This says to get the parents except the IdentityBlock
        super(IdentityBlock, self).__init__(name='')

        self.conv1 = Conv2D(filters, kernel_size, padding='same')
        self.bn1 = BatchNormalization()

        self.conv2 = Conv2D(filters, kernel_size, padding='same')
        self.bn2 = BatchNormalization()

        self.act = Activation('relu')
        self.add = Add()

    # The input tensor is the input supplied to the block
    def call(self, input_tensor):
        x = self.conv1(input_tensor)
        x = self.bn1(x)
        x = self.act(x)
        x = self.conv2(x)
        x = self.bn2(x)

        # Addition of (processed data & input data)
        x = self.add([x, input_tensor])
        x = self.act(x)
        return x

### Build the ResNet Model

In [5]:
class MiniResNet(tf.keras.Model):
    # Number of classes to be classified into
    def __init__(self, num_classes):
        super(MiniResNet, self).__init__()
        self.conv = Conv2D(64, 7, padding="same")
        self.bn = BatchNormalization()
        self.act = Activation('relu')
        self.max_pool = MaxPooling2D((3,3))

        # Define the identity blocks
        self.idblk1 = IdentityBlock(64, 3)
        self.idblk2 = IdentityBlock(64, 3)

        self.global_avg_pool = GlobalAvgPool2D()
        self.classifier = tf.keras.layers.Dense(num_classes, activation="softmax")

    def call(self, inputs):
        x = self.conv(inputs)
        x = self.bn(x)
        x = self.act(x)
        x = self.max_pool(x)

        x = self.idblk1(x)
        x = self.idblk2(x)

        x = self.global_avg_pool(x)
        return self.classifier(x)

### Train the model

In [7]:
# Utility function to normalize images and return (image,label) pairs
def preprocess(features):
    return tf.cast(features['image'], tf.float32) / 255., features['label']

resnet = MiniResNet(10)
resnet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

dataset = tfds.load('mnist', split=tfds.Split.TRAIN, data_dir='./data')
dataset = dataset.map(preprocess).batch(32)

resnet.fit(dataset, epochs=10)

2026-02-12 22:42:23.467807: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-02-12 22:42:23.468032: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-02-12 22:42:23.468039: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1770916343.468504   34739 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1770916343.469369   34739 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling data/mnist/incomplete.UFUPSS_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/10000 [00:00<?, ? exa…

Dataset mnist downloaded and prepared to data/mnist/3.0.1. Subsequent calls will reuse this data.
Epoch 1/10


2026-02-12 22:43:34.430363: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-02-12 22:43:35.003227: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:387] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 72s 35ms/step - accuracy: 0.7631 - loss: 0.6786
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 67s 36ms/step - accuracy: 0.9777 - loss: 0.0777
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 67s 36ms/step - accuracy: 0.9841 - loss: 0.0530
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 66s 35ms/step - accuracy: 0.9877 - loss: 0.0413
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 66s 35ms/step - accuracy: 0.9906 - loss: 0.0328
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 66s 35ms/step - accuracy: 0.9921 - loss: 0.0272
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 65s 35ms/step - accuracy: 0.9929 - loss: 0.0228
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 65s 35ms/step - accuracy: 0.9939 - loss: 0.0209
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 65s 35ms/step - accuracy: 0.9954 - loss: 0.0169
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 66s 35ms/step - accuracy: 0.9959 - loss: 0.0148
